### This notebook will clean the data from year 2016
#### The cleaning is done step-by-step in comprehensible steps and small comments/visualisations for each step

Input: `data/processed/extracted/2016.csv`. Output: `data/processed/clean_2016.csv` (1,431 rows x 138 columns).

In [18]:
import sys; sys.path.append("../src")
import pandas as pd
from cleaning import AGE_MIN, AGE_MAX, MALE, FEMALE, UNUSABLE, UNSURE_MAP, ORDER_EMP, ORDER_WORK

raw = pd.read_csv("../data/processed/extracted/2016.csv", keep_default_na=False, na_values=[""])
# read the data, keep N/A as there were some answers like N/A which are different from the empty cells

print(raw.shape)    # (1433, 63)
display(raw.head())

(1433, 63)


,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",...,"If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?","If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?",What is your age?,What is your gender?,What country do you live in?,What US state or territory do you live in?,What country do you work in?,What US state or territory do you work in?,Which of the following best describes your work position?,Do you work remotely?
0,0,26-100,1.0,NaN,Not eligible for coverage / N/A,N/A,No,No,I don't know,Very easy,...,Not applicable to me,Not applicable to me,39,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Sometimes
1,0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,Rarely,Sometimes,29,male,United States of America,Illinois,United States of America,Illinois,Back-end Developer|Front-end Developer,Never
2,0,6-25,1.0,NaN,No,N/A,No,No,I don't know,Neither easy nor difficult,...,Not applicable to me,Not applicable to me,38,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Always
3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Sometimes,Sometimes,43,male,United Kingdom,NaN,United Kingdom,NaN,Supervisor/Team Lead,Sometimes
4,0,6-25,0.0,1.0,Yes,Yes,No,No,No,Neither easy nor difficult,...,Sometimes,Sometimes,43,Female,United States of America,Illinois,United States of America,Illinois,Executive Leadership|Supervisor/Team Lead|Dev ...,Sometimes


### Here we can see several NaN (empty cells) and multiple-choice questions (separated by "|" delimiter)

Renaming the columns - will be better to navigate, also to match the data with 2014 and later datasets; Here the table is to cross-check the results of the current naming;

In [19]:
RENAME = [
    "self_employed", "no_employees", "tech_company", "tech_role", "benefits", "care_options",
    "wellness_program", "seek_help", "anonymity", "leave", "mental_health_consequence",
    "phys_health_consequence", "coworkers", "supervisor", "mental_vs_physical", "obs_consequence",
    "selfemp_medical_coverage", "selfemp_know_resources", "selfemp_reveal_clients",
    "selfemp_reveal_clients_negative", "selfemp_reveal_coworkers",
    "selfemp_reveal_coworkers_negative", "selfemp_productivity_affected",
    "selfemp_productivity_pct_affected",
    "has_previous_employers", "prev_benefits", "prev_care_options",
    "prev_wellness_program", "prev_seek_help", "prev_anonymity",
    "prev_mental_health_consequence", "prev_phys_health_consequence", "prev_coworkers",
    "prev_supervisor", "prev_mental_vs_physical", "prev_obs_consequence",
    "phys_health_interview", "phys_health_interview_why",
    "mental_health_interview", "mental_health_interview_why", "career_hurt",
    "coworkers_view_negatively", "share_friends_family", "unsupportive_response",
    "observation_deterred_disclosure", "family_history", "past_disorder", "current_disorder",
    "current_yes_conditions", "current_maybe_conditions", "diagnosed_by_professional",
    "professional_diagnosis_conditions", "treatment", "work_interfere_treated",
    "work_interfere_untreated", "Age", "Gender", "Country", "state", "Country_work",
    "state_work", "work_position", "remote_work",
]

In [20]:
overview = pd.DataFrame({"Renamed": RENAME, "current": raw.columns})

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(overview)

,Renamed,current
0,self_employed,Are you self-employed?
1,no_employees,How many employees does your company or organization have?
2,tech_company,Is your employer primarily a tech company/organization?
3,tech_role,Is your primary role within your company related to tech/IT?
4,benefits,Does your employer provide mental health benefits as part of healthcare coverage?
5,care_options,Do you know the options for mental health care available under your employer-provided coverage?
6,wellness_program,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?"
7,seek_help,Does your employer offer resources to learn more about mental health concerns and options for seeking help?
8,anonymity,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?
9,leave,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:"


In [21]:
# data observation, to then put it into categorical/Boolean/multiple choice;
overview = pd.DataFrame({"dtype": raw.dtypes, 
                         "unique": raw.nunique(),
                         "missing": raw.isna().sum()}).reset_index(names="question")

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(overview)

,question,dtype,unique,missing
0,Are you self-employed?,int64,2,0
1,How many employees does your company or organization have?,str,6,287
2,Is your employer primarily a tech company/organization?,float64,2,287
3,Is your primary role within your company related to tech/IT?,float64,2,1170
4,Does your employer provide mental health benefits as part of healthcare coverage?,str,4,287
5,Do you know the options for mental health care available under your employer-provided coverage?,str,4,287
6,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",str,3,287
7,Does your employer offer resources to learn more about mental health concerns and options for seeking help?,str,3,287
8,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,str,3,287
9,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",str,6,287


For now, up to 10 values - should be mostly categorical/boolean, many answers - either multiple choice, identity classifying columns in broader sense;

So, will investigate each one and then change their type(make a map) for most of them

In [22]:
for col in raw.columns:
    if raw[col].nunique() <= 10:
        #print(f"--- {col}")
        print(raw[col].value_counts(dropna=False))
        print()

Are you self-employed?
0    1146
1     287
Name: count, dtype: int64

How many employees does your company or organization have?
26-100            292
NaN               287
More than 1000    256
100-500           248
6-25              210
500-1000           80
1-5                60
Name: count, dtype: int64

Is your employer primarily a tech company/organization?
1.0    883
NaN    287
0.0    263
Name: count, dtype: int64

Is your primary role within your company related to tech/IT?
NaN    1170
1.0     248
0.0      15
Name: count, dtype: int64

Does your employer provide mental health benefits as part of healthcare coverage?
Yes                                531
I don't know                       319
NaN                                287
No                                 213
Not eligible for coverage / N/A     83
Name: count, dtype: int64

Do you know the options for mental health care available under your employer-provided coverage?
No               354
I am not sure    352
Yes     

## From this we can see the categories
### Mostly separation is into Boolean and Categorical, also not applicable should be a separate option

Mostly they have a similar structure, and we will reuse the cleaning.py module for that

"Not applicable" stays as a value in the same column; in the score columns it is left blank.

## Cleaning: rename, Yes/No, age

In [23]:
# copy raw, rename columns, 0/1 -> Yes/No, remove impossible ages
df = raw.copy()
df.columns = RENAME

for c in ["self_employed", "tech_company", "tech_role", "selfemp_medical_coverage",
          "has_previous_employers", "treatment"]:
    df[c] = df[c].map({1: "Yes", 0: "No"})

df = df[df["Age"].between(AGE_MIN, AGE_MAX)].copy()

## Gender

Symbols used below. Sets: `|` combines, `&` keeps items in both, `-` removes items, `|=` adds items. pandas True/False values: `~` means *not*.

In [24]:
# gender mapping
OTHER = {
    # non-binary, queer or similar
    "non-binary", "nonbinary", "genderqueer", "agender", "enby", "genderfluid", "fluid",
    "queer", "androgynous", "bigender", "genderflux demi-girl", "genderqueer woman",
    "male/genderqueer", "nb masculine",
    # trans
    "transgender woman", "male (trans, ftm)", "mtf", "transitioned, m2f",
    "other/transfeminine",
    # unclear : "fm" - female/femme/etc. - ?
    "fm", "female-bodied; no feelings about gender",
    "female or multi-gender femme", "genderfluid (born female)",
    "male 9:1 female, roughly", "other",
    # "afab" answers are in FEMALE (cleaning.py): my assumption, as they don't state current identity
}

known = MALE | FEMALE | OTHER | UNUSABLE             # all spellings that have a group
gender_clean = df["Gender"].dropna().str.strip().str.lower()
unmatched = gender_clean[~gender_clean.isin(known)]  # keep those NOT in known
print(unmatched.value_counts())                      # should be empty

Series([], Name: count, dtype: int64)


In [25]:
# uses the lists from cleaning.py and OTHER; an unlisted answer raises an error
def group_gender(g):
    if not isinstance(g, str):
        return None
    g = g.strip().lower()
    if g in MALE:     return "male"
    if g in FEMALE:   return "female"
    if g in OTHER:    return "other"
    if g in UNUSABLE: return None
    raise ValueError(f"unmapped gender: {g!r}")

df["Gender"] = df["Gender"].map(group_gender)
print(df["Gender"].value_counts(dropna=False))

Gender
male      1055
female     338
other       31
NaN          7
Name: count, dtype: int64


## Unsure and not applicable

In [26]:
# only relabel choice questions; free-text answers stay as written
text_cols = ["phys_health_interview_why", "mental_health_interview_why"]
choice_cols = df.columns.difference(text_cols)

df[choice_cols] = df[choice_cols].replace(UNSURE_MAP)
df[choice_cols] = df[choice_cols].replace(
    {"N/A": "Not applicable", "Not applicable to me": "Not applicable"})
print(df.shape)          # (1431, 63)

(1431, 63)


## Ordered scores

Categorical values lists for remapping - for some of them we were able to transfer the old mappings;

In [27]:
HOW_MANY = ["None did", "Some did", "Yes, they all did"]
OF_THEM = ["None of them", "Some of them", "Yes, all of them"]
AT_EMP = ["No, at none of my previous employers", "Some of my previous employers",
          "Yes, at all of my previous employers"]

ORDINAL = {
    "no_employees": ORDER_EMP,
    "remote_work": ["Never", "Sometimes", "Always"],
    "work_interfere_treated": ORDER_WORK,
    "work_interfere_untreated": ORDER_WORK,
    "leave": ["Very easy", "Somewhat easy", "Neither easy nor difficult",
              "Somewhat difficult", "Very difficult"],
    "selfemp_know_resources": ["No, I don't know any", "I know some", "Yes, I know several"],
    "selfemp_productivity_pct_affected": ["1-25%", "26-50%", "51-75%", "76-100%"],
    "share_friends_family": ["Not open at all", "Somewhat not open", "Neutral",
                             "Somewhat open", "Very open"],
    "career_hurt": ["No, it has not", "No, I don't think it would", "Unsure",
                    "Yes, I think it would", "Yes, it has"],
    "coworkers_view_negatively": ["No, they do not", "No, I don't think they would", "Unsure",
                                  "Yes, I think they would", "Yes, they do"],
    "prev_benefits": ["No, none did", "Some did", "Yes, they all did"],
    "prev_wellness_program": HOW_MANY,
    "prev_seek_help": HOW_MANY,
    "prev_mental_vs_physical": HOW_MANY,
    "prev_mental_health_consequence": OF_THEM,
    "prev_phys_health_consequence": OF_THEM,
    "prev_obs_consequence": OF_THEM,
    "prev_coworkers": AT_EMP,
    "prev_supervisor": AT_EMP,
    "prev_anonymity": ["No", "Sometimes", "Yes, always"],
}
OFF_SCALE = {"Unsure", "Not applicable",
             "Not applicable to me (I do not have a mental illness)"}

for col, order in ORDINAL.items():
    # answers that are neither on the scale nor listed as off-scale; must be none
    unexpected = set(df[col].dropna()) - set(order) - OFF_SCALE
    assert not unexpected, (col, unexpected)
    # position in the list = score (0, 1, 2...); answers not in the list become blank
    df[col + "_score"] = df[col].map({answer: i for i, answer in enumerate(order)})

## Multiple-choice questions

- Work position: one 0/1 column per role.
- Conditions: the 12 fixed survey options; typed-in answers are grouped manually.
- Autism spectrum answers have their own group. `other` = the remaining typed-in answers (not necessarily a disorder).
- A blank condition column means there is no condition list for that person, not "no".

In [28]:
# quick look at the table before adding the multiple-choice columns
df.head()

,self_employed,no_employees,tech_company,tech_role,benefits,care_options,wellness_program,seek_help,anonymity,leave,...,prev_benefits_score,prev_wellness_program_score,prev_seek_help_score,prev_mental_vs_physical_score,prev_mental_health_consequence_score,prev_phys_health_consequence_score,prev_obs_consequence_score,prev_coworkers_score,prev_supervisor_score,prev_anonymity_score
0,No,26-100,Yes,NaN,Not eligible for coverage / N/A,Not applicable,No,No,Unsure,Very easy,...,0.0,NaN,0.0,NaN,1.0,0.0,0.0,1.0,1.0,NaN
1,No,6-25,Yes,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,2.0
2,No,6-25,Yes,NaN,No,Not applicable,No,No,Unsure,Neither easy nor difficult,...,0.0,0.0,1.0,NaN,NaN,1.0,1.0,1.0,NaN,NaN
3,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,0.0,0.0,NaN,1.0,1.0,1.0,1.0,1.0,NaN
4,No,6-25,No,Yes,Yes,Yes,No,No,No,Neither easy nor difficult,...,NaN,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,NaN


In [29]:
# split "A|B" and put each choice on its own row
roles = df["work_position"].str.split("|").explode()
print(roles.value_counts())

ROLE_NAMES = {
    "Back-end Developer": "backend", "Front-end Developer": "frontend",
    "DevOps/SysAdmin": "devops_sysadmin", "Supervisor/Team Lead": "supervisor_lead",
    "Other": "other", "Support": "support", "One-person shop": "one_person_shop",
    "Designer": "designer", "Executive Leadership": "executive",
    "Dev Evangelist/Advocate": "dev_advocate", "Sales": "sales", "HR": "hr",
}
unmatched = set(roles.dropna()) - set(ROLE_NAMES)
assert not unmatched, unmatched

chosen_roles = df["work_position"].str.split("|")
# one column per role: 1 = selected, 0 = not selected
for answer, name in ROLE_NAMES.items():
    df["role_" + name] = chosen_roles.map(lambda options: int(answer in options))

work_position
Back-end Developer         735
Front-end Developer        502
DevOps/SysAdmin            282
Supervisor/Team Lead       277
Other                      187
Support                    168
One-person shop            161
Designer                   135
Executive Leadership       101
Dev Evangelist/Advocate     99
Sales                       31
HR                          12
Name: count, dtype: int64


In [30]:
CONDITION_COLS = {"current_yes_conditions": "cur_",
                  "current_maybe_conditions": "maybe_",
                  "professional_diagnosis_conditions": "diag_"}

# the 12 fixed options of the survey
CONDITIONS = {
    "mood": {"Mood Disorder (Depression, Bipolar Disorder, etc)"},
    "anxiety": {"Anxiety Disorder (Generalized, Social, Phobia, etc)"},
    "adhd": {"Attention Deficit Hyperactivity Disorder"},
    "ptsd": {"Post-traumatic Stress Disorder"},
    "ocd": {"Obsessive-Compulsive Disorder"},
    "stress_response": {"Stress Response Syndromes"},
    "personality": {"Personality Disorder (Borderline, Antisocial, Paranoid, etc)"},
    "substance_use": {"Substance Use Disorder"},
    "addictive": {"Addictive Disorder"},
    "eating": {"Eating Disorder (Anorexia, Bulimia, etc)"},
    "dissociative": {"Dissociative Disorder"},
    "psychotic": {"Psychotic Disorder (Schizophrenia, Schizoaffective, etc)"},
}

# every selected or typed condition from all three columns
all_options = pd.concat(
    [df[c].dropna().str.split("|").explode() for c in CONDITION_COLS]).str.strip()
# combine all groups' strings into one set
fixed = set().union(*CONDITIONS.values())
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(all_options[~all_options.isin(fixed)].value_counts())   # typed-in answers

Autism                                                                                                                                                       6
Depression                                                                                                                                                   4
Seasonal Affective Disorder                                                                                                                                  2
Burn out                                                                                                                                                     2
PDD-NOS                                                                                                                                                      2
Autism (Asperger's)                                                                                                                                          2
Gender Dysphoria                              

In [31]:
# decisions for the typed-in answers (one answer may be in two groups)
MOOD_AND_ANXIETY = {"post-partum / anxiety",
        "I haven't been formally diagnosed, so I felt uncomfortable answering, but Social Anxiety and Depression."}
CONDITIONS["mood"] |= MOOD_AND_ANXIETY | {"Depression", "Seasonal Affective Disorder"}
CONDITIONS["anxiety"] |= MOOD_AND_ANXIETY
CONDITIONS["adhd"] |= {"ADD (w/o Hyperactivity)", "Attention Deficit Disorder",
                       "attention deficit disorder (but not the hyperactive version)"}
CONDITIONS["ptsd"] |= {"posttraumatic stress disourder", "PTSD (undiagnosed)"}
CONDITIONS["personality"] |= {"Schizotypal Personality Disorder"}
CONDITIONS["addictive"] |= {"Sexual addiction"}
CONDITIONS["dissociative"] |= {"Depersonalisation", "depersonalization disorder"}
CONDITIONS["autism_spectrum"] = { # autism spectrum
    "Autism", "Autism (Asperger's)", "Autism Spectrum Disorder", "Autism spectrum disorder",
    "autism spectrum disorder", "Asperger's", "Aspergers", "Asperger Syndrome", "Asperges",
    "PDD-NOS", "PDD-NOS (see above)", "Pervasive Developmental Disorder (Not Otherwise Specified)",
    """Autism - while not a "mental illness", still greatly affects how I handle anxiety""",}
CONDITIONS["other"] = {    
    # everything else
    "Burn out", "Burnout", "Suicidal Ideation", "Gender Dysphoria", "Gender Identity Disorder",
    "Transgender", "Intimate Disorder", "Traumatic Brain Injury", "Sleeping Disorder",
    "Tinnitus", "We're all hurt, right?!",
    """MCD (when it was diagnosed, the ultra-mega "disorder" ADHD didn't exist yet)""",
    """Combination of physical impairment (strongly near-sighted) with a possibly mental one (MCD / "ADHD", though its actually a stimulus filtering impairment)""",
}

unmatched = set(all_options) - set().union(*CONDITIONS.values())
assert not unmatched, unmatched

In [32]:
def has_condition(cell_text, group_answers):
    """1 = an answer from this group was selected, 0 = not, blank = no condition list."""
    if pd.isna(cell_text):
        return pd.NA
    chosen = {answer.strip() for answer in cell_text.split("|")}
    found = chosen & group_answers             # answers present in both sets
    return int(len(found) > 0)

# one column for each combination of source question and condition group
new_columns = {}
for source, prefix in CONDITION_COLS.items():
    for group, group_answers in CONDITIONS.items():
        flags = [has_condition(text, group_answers) for text in df[source]]
        new_columns[prefix + group] = pd.array(flags, dtype="Int64")   # integers that allow blanks
new_flags = pd.DataFrame(new_columns, index=df.index)

df = df.drop(columns=new_flags.columns, errors="ignore")   # avoids duplicates on a re-run
df = pd.concat([df, new_flags], axis=1)                    # add the new columns on the right

for source, prefix in CONDITION_COLS.items():
    flags = [prefix + group for group in CONDITIONS]
    print(source, "| answered:", df[source].notna().sum(),
          "| rows with a flag:", int((df[flags].sum(axis=1) > 0).sum()))
    print(df[flags].sum().to_string())
    print()

current_yes_conditions | answered: 568 | rows with a flag: 568
cur_mood               415
cur_anxiety            346
cur_adhd               122
cur_ptsd                70
cur_ocd                 45
cur_stress_response     38
cur_personality         36
cur_substance_use       39
cur_addictive           36
cur_eating              28
cur_dissociative        11
cur_psychotic            6
cur_autism_spectrum      9
cur_other                7

current_maybe_conditions | answered: 321 | rows with a flag: 321
maybe_mood               220
maybe_anxiety            192
maybe_adhd                58
maybe_ptsd                26
maybe_ocd                 30
maybe_stress_response     40
maybe_personality         42
maybe_substance_use       16
maybe_addictive           24
maybe_eating              10
maybe_dissociative         7
maybe_psychotic            8
maybe_autism_spectrum      3
maybe_other                5

professional_diagnosis_conditions | answered: 710 | rows with a flag: 710
diag_mood   

## Blank checks

Follow-up questions should be empty for people who were not asked.

In [33]:
def check(asked, cols, label):
    extra = int(df.loc[~asked, cols].notna().sum().sum())     # answered although not asked
    blank = int(df.loc[asked, cols].isna().sum().sum())       # asked but blank
    print(f"{label:36} answered-not-asked: {extra:3}   asked-but-blank: {blank}")

# RENAME follows the survey's question order, so each slice is one block
employer_cols = RENAME[1:3] + RENAME[4:16]   # no_employees ... obs_consequence, without tech_role
selfemp_cols = RENAME[16:23]                 # selfemp_medical_coverage ... selfemp_productivity_affected
prev_cols = RENAME[25:36]                    # prev_benefits ... prev_obs_consequence

check(df["self_employed"] == "No", employer_cols, "employer block")
check(df["self_employed"] == "Yes", selfemp_cols, "self-employed block")
check(df["has_previous_employers"] == "Yes", prev_cols, "previous-employer block")
check(df["tech_company"] == "No", ["tech_role"], "tech_role")
check(df["selfemp_productivity_affected"] == "Yes", ["selfemp_productivity_pct_affected"], "productivity %")
check(df["current_disorder"] == "Yes", ["current_yes_conditions"], "current_yes_conditions")
check(df["current_disorder"] == "Unsure", ["current_maybe_conditions"], "current_maybe_conditions")
check(df["diagnosed_by_professional"] == "Yes", ["professional_diagnosis_conditions"], "diagnosis conditions")
check(df["unsupportive_response"].isin(["Yes, I observed", "Yes, I experienced", "Unsure"]),
      ["observation_deterred_disclosure"], "observation_deterred_disclosure")
check(df["Country"] == "United States of America", ["state"], "state")
check(df["Country_work"] == "United States of America", ["state_work"], "state_work")

employer block                       answered-not-asked:   0   asked-but-blank: 0
self-employed block                  answered-not-asked:   0   asked-but-blank: 0
previous-employer block              answered-not-asked:   0   asked-but-blank: 0
tech_role                            answered-not-asked:   0   asked-but-blank: 0
productivity %                       answered-not-asked:   0   asked-but-blank: 0
current_yes_conditions               answered-not-asked:   0   asked-but-blank: 7
current_maybe_conditions             answered-not-asked:   0   asked-but-blank: 5
diagnosis conditions                 answered-not-asked:   0   asked-but-blank: 5
observation_deterred_disclosure      answered-not-asked:   0   asked-but-blank: 2
state                                answered-not-asked:   0   asked-but-blank: 0
state_work                           answered-not-asked:   0   asked-but-blank: 0


## Save

In [34]:
df["year"] = 2016
df = df.reset_index(drop=True)
df.to_csv("../data/processed/clean_2016.csv", index=False)

# read it back the same way as the raw file, so text such as "NA" is not turned into NaN
saved = pd.read_csv("../data/processed/clean_2016.csv", keep_default_na=False, na_values=[""])
print(df.shape, saved.shape)
assert saved.shape == df.shape                         # same rows and columns after reloading
assert (saved.isna().sum() == df.isna().sum()).all()   # same number of blanks in each column

(1431, 138) (1431, 138)
